# 02 · Train Encoders — Multi-model comparison

This notebook is the **orchestrator**: it imports all business logic from `src/`
and loops over models × augmentation strategies × loss functions.

All results are saved to `results/` as JSON and can be compared at the end
(or in `03_compare_results.ipynb`).

---
### Quick-start
1. Adjust `MODELS_TO_RUN`, `AUGMENTATIONS`, `LOSSES` in the **Configuration** cell.
2. Run all cells top to bottom.
3. Results appear in `results/` and in the final comparison table.

## 0 · Setup (Colab only)

In [ ]:
# ── Google Colab setup ────────────────────────────────────────────────────────
# This cell is a no-op when running locally.
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Mount Drive to persist results across sessions (optional)
    from google.colab import drive
    drive.mount('/content/drive')

    # Clone the project repo if running directly from Colab
    # !git clone https://github.com/YOUR_USER/YOUR_REPO.git
    # %cd YOUR_REPO

    # Install dependencies
    !pip install -q transformers datasets scikit-learn torch nltk
    print('Colab dependencies installed.')
else:
    print('Running locally — skipping Colab setup.')

In [ ]:
# Add project root to sys.path so `src/` is importable from any working directory
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/progettoLLM/project').resolve()   # notebooks/ is one level below root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)

## 1 · Configuration

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# EDIT THIS CELL to control what gets trained.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from config.model_configs import MODEL_CONFIGS
from src.data.augmentation import list_augmentations

# ── Models to benchmark ───────────────────────────────────────────────────────
# Comment out any model you don't want to run.
MODELS_TO_RUN = [
    'distilbert-base-uncased',
    'longformer-base',              # original notebook baseline
    #'deberta-v3-base',              # paper baseline — strong encoder
    'roberta-base',                 # paper baseline
    # 'xlnet-base',                 # paper baseline — slower, enable if needed
    # 'longformer-large',           # needs >20 GB VRAM

    # 'albert-base-v2',
     'bert-base-uncased',          # reference

]

# ── Augmentation strategies ───────────────────────────────────────────────────
# Available: print(list_augmentations())
AUGMENTATIONS = [
    # 'none',                  # control: no augmentation
    # 'synonym_replacement',   # EDA-style synonym swap on minority classes
    # 'oversampling',          # duplicate minority examples to balance
    # 'random_deletion',
    # 'random_swap',
    # 'back_translation',    # slow — needs GPU and internet
    # 'length_category',    # adds a length-based feature (ablation)
     'tone_analysis',       # adds a tone-based feature (ablation)
]

# ── Loss functions ────────────────────────────────────────────────────────────
# Available: 'focal', 'weighted_ce', 'label_smoothing', 'ce'
LOSSES = [
    #'focal',         # original notebook loss
    # 'weighted_ce', # ablation: class-weighted CE without focal modulation
     'ce',          # ablation: vanilla cross-entropy (no weighting)
]

# ── Shared hyperparameters ────────────────────────────────────────────────────
NUM_EPOCHS    = 5
WEIGHT_DECAY  = 0.01
FOCAL_GAMMA   = 2.0
SEED          = 42

# ── Task setting ──────────────────────────────────────────────────────────────
# 'evasion'  → 9-class fine-grained task  (used in the original notebook)
# 'clarity'  → 3-class high-level task    (Clear Reply / Ambivalent / Clear Non-Reply)
TASK = 'clarity'

print('Models     :', MODELS_TO_RUN)
print('Augment.   :', AUGMENTATIONS)
print('Losses     :', LOSSES)
print('Total runs :', len(MODELS_TO_RUN) * len(AUGMENTATIONS) * len(LOSSES))

## 2 · Load dataset (once — shared by all runs)

In [ ]:
from src.data.dataset_loader import load_and_split_dataset

print('Loading ailsntua/QEvasion ...')
raw_ds = load_and_split_dataset(seed=SEED, verbose=True)
raw_ds

## 3 · Build label maps (once)

In [ ]:
from src.data.label_utils import build_label_maps, compute_alpha_weights, apply_labels

# Select the label column based on the task
import src.data.label_utils as lu
if TASK == 'clarity':
    lu.LABEL_COLUMN = 'clarity_label'   # switch to 3-class task
else:
    lu.LABEL_COLUMN = 'evasion_label'   # default 9-class task

label2id, id2label = build_label_maps(raw_ds)
num_classes = len(label2id)

print(f'Task          : {TASK}')
print(f'Num classes   : {num_classes}')
print(f'Label mapping : {label2id}')

# Compute inverse-frequency weights (for FocalLoss / WeightedCE)
alpha_tensor = compute_alpha_weights(raw_ds, label2id)
print('\nAlpha weights (inverse frequency):')
for i, w in enumerate(alpha_tensor.tolist()):
    print(f'  {id2label[i]:<25} : {w:.4f}')

# Apply integer labels to all splits
labeled_ds = apply_labels(raw_ds, label2id, verbose=True)

## 4 · Main training loop

In [ ]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from config.model_configs import get_model_config
from src.data.augmentation import get_augmentation_fn
from src.training.metrics import compute_metrics, compute_detailed_report
from src.training.trainers import get_trainer
from src.utils.env_utils import get_training_args
from src.utils.results_utils import save_results


def free_memory():
    """Release GPU/CPU memory between runs."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def tokenize_for_model(ds, tokenizer, model_cfg: dict):
    """Tokenize the dataset for a specific model configuration."""
    max_length = model_cfg['max_length']
    use_token_type_ids = model_cfg.get('token_type_ids', False)

    def tokenize_fn(examples):
        return tokenizer(
            examples['question'],
            examples['interview_answer'],
            padding='max_length',
            truncation=True,
            max_length=max_length,
        )

    tokenized = ds.map(tokenize_fn, batched=True)

    # Keep only the columns the model actually needs
    columns_to_keep = ['input_ids', 'attention_mask', 'label']
    if use_token_type_ids:
        columns_to_keep.append('token_type_ids')

    # Some models (e.g. Longformer) also produce global_attention_mask
    if 'global_attention_mask' in tokenized['train'].column_names:
        columns_to_keep.append('global_attention_mask')

    tokenized.set_format(type='torch', columns=columns_to_keep)
    return tokenized


# ── Outer loop ────────────────────────────────────────────────────────────────
all_run_results = []

for model_key in MODELS_TO_RUN:
    model_cfg = get_model_config(model_key)
    model_id  = model_cfg['model_id']

    print(f'\n{"="*70}')
    print(f'MODEL : {model_key}  ({model_id})')
    print(f'{"="*70}')

    # Load tokenizer once per model
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    for aug_name in AUGMENTATIONS:
        print(f'\n  ── Augmentation: {aug_name} ──')

        # Apply augmentation to training split
        aug_fn = get_augmentation_fn(aug_name)
        aug_ds = aug_fn(labeled_ds, label2id, seed=SEED)
        print(f'     Train size after augmentation: {len(aug_ds["train"])}')

        # Tokenize for this model
        tokenized_ds = tokenize_for_model(aug_ds, tokenizer, model_cfg)

        for loss_name in LOSSES:
            print(f'\n    ── Loss: {loss_name} ──')

            run_tag = f'{model_key} / {aug_name} / {loss_name}'

            # ── Build model (fresh weights each run) ──────────────────────────
            model = AutoModelForSequenceClassification.from_pretrained(
                model_id,
                num_labels=num_classes,
                id2label=id2label,
                label2id=label2id,
                ignore_mismatched_sizes=True,
            )

            # ── Build TrainingArguments ───────────────────────────────────────
            training_args = get_training_args(
                model_key=model_key,
                model_cfg=model_cfg,
                aug_name=aug_name,
                loss_name=loss_name,
                num_epochs=NUM_EPOCHS,
                weight_decay=WEIGHT_DECAY,
            )

            # ── Build Trainer ─────────────────────────────────────────────────
            trainer = get_trainer(
                loss_name=loss_name,
                model=model,
                training_args=training_args,
                train_dataset=tokenized_ds['train'],
                eval_dataset=tokenized_ds['validation'],
                compute_metrics=compute_metrics,
                alpha=alpha_tensor,
                gamma=FOCAL_GAMMA,
                num_classes=num_classes,
            )

            # ── Train ─────────────────────────────────────────────────────────
            print(f'    Starting training for: {run_tag}')
            trainer.train()

            # ── Evaluate on VALIDATION set ────────────────────────────────────
            val_metrics = trainer.evaluate()
            print(f'    Validation metrics: {val_metrics}')

            # ── Evaluate on TEST set ──────────────────────────────────────────
            test_pred = trainer.predict(tokenized_ds['test'])
            test_metrics = test_pred.metrics
            print(f'    Test metrics: {test_metrics}')

            # Detailed per-class report
            report = compute_detailed_report(
                test_pred.predictions,
                test_pred.label_ids,
                id2label,
            )
            print(f'\n    Per-class report:\n{report}')

            # ── Save results ──────────────────────────────────────────────────
            save_results(
                model_key=model_key,
                aug_name=aug_name,
                loss_name=loss_name,
                metrics={
                    'val_macro_f1':      val_metrics.get('eval_macro_f1', 0),
                    'val_weighted_f1':   val_metrics.get('eval_weighted_f1', 0),
                    'val_accuracy':      val_metrics.get('eval_accuracy', 0),
                    'test_macro_f1':     test_metrics.get('test_macro_f1', 0),
                    'test_weighted_f1':  test_metrics.get('test_weighted_f1', 0),
                    'test_accuracy':     test_metrics.get('test_accuracy', 0),
                },
                extra={
                    'model_id':    model_id,
                    'max_length':  model_cfg['max_length'],
                    'num_epochs':  NUM_EPOCHS,
                    'focal_gamma': FOCAL_GAMMA,
                    'task':        TASK,
                    'train_size':  len(aug_ds['train']),
                },
            )
            all_run_results.append(run_tag)

            # ── Clean up before next run ──────────────────────────────────────
            del model, trainer
            free_memory()

        del tokenized_ds
        free_memory()

    del tokenizer
    free_memory()

print('\n\nAll runs completed:')
for r in all_run_results:
    print(' ✓', r)

## 5 · Quick comparison table

In [ ]:
from src.utils.results_utils import compare_results, print_comparison_table

df = compare_results(
    metrics_to_show=['test_macro_f1', 'test_weighted_f1', 'test_accuracy',
                     'val_macro_f1']
)
df

In [ ]:
# Rich table (if rich is installed, otherwise plain print)
print_comparison_table()